In [1]:
import pygame
import random
import math
from collections import deque

pygame.init()

WIDTH = 600
HEIGHT = 600
CELL = 30

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("AI Pac-Man")

clock = pygame.time.Clock()
font = pygame.font.SysFont(None, 36)

maze = [
"11111111111111111111",
"10000000001000000001",
"10111111101011111101",
"10000000000000000001",
"10111101111101111101",
"10000001000001000001",
"11110101111010101111",
"10000100000000100001",
"10111111101111111101",
"10000000000000000001",
"11111111111111111111"
]

pacman = [1,1]
score = 0

power_mode = False
power_timer = 0

move_delay = 0
ghost_delay = 0

mouth_angle = 0
mouth_dir = 1

pellets = []
power_pellets = []

for y,row in enumerate(maze):
    for x,col in enumerate(row):
        if col == "0":
            pellets.append((x,y))

power_pellets = random.sample(pellets,4)

ghosts=[]
ghost_colors=[(255,0,0),(255,105,180),(0,255,255)]

for i in range(3):
    while True:
        x=random.randint(1,18)
        y=random.randint(1,9)
        if maze[y][x]=="0":
            ghosts.append({"pos":[x,y]})
            break


def neighbors(node):

    x,y=node
    dirs=[(1,0),(-1,0),(0,1),(0,-1)]

    result=[]

    for dx,dy in dirs:

        nx=x+dx
        ny=y+dy

        if maze[ny][nx]=="0":
            result.append((nx,ny))

    return result


def bfs(start,goals):

    queue=deque([[start]])
    visited=set()

    while queue:

        path=queue.popleft()
        node=path[-1]

        if node in goals:
            return path

        if node not in visited:

            visited.add(node)

            for n in neighbors(node):

                new=list(path)
                new.append(n)
                queue.append(new)

    return None


def move_pacman():

    global score,power_mode,power_timer

    nearest_dist=999
    nearest_ghost=None

    for g in ghosts:

        d=abs(g["pos"][0]-pacman[0])+abs(g["pos"][1]-pacman[1])

        if d<nearest_dist:
            nearest_dist=d
            nearest_ghost=g

    # escape if ghost is very close
    if nearest_dist<=2 and not power_mode:

        dirs=[(1,0),(-1,0),(0,1),(0,-1)]

        best=None
        best_dist=-1

        for dx,dy in dirs:

            nx=pacman[0]+dx
            ny=pacman[1]+dy

            if maze[ny][nx]=="1":
                continue

            dist=abs(nx-nearest_ghost["pos"][0])+abs(ny-nearest_ghost["pos"][1])

            if dist>best_dist:
                best_dist=dist
                best=(nx,ny)

        if best:
            pacman[0],pacman[1]=best

    else:

        if pellets:

            path=bfs(tuple(pacman),pellets)

            if path and len(path)>1:
                pacman[0],pacman[1]=path[1]

    if tuple(pacman) in pellets:
        pellets.remove(tuple(pacman))
        score+=10

    if tuple(pacman) in power_pellets:
        power_pellets.remove(tuple(pacman))
        power_mode=True
        power_timer=200


def move_ghost(g):

    x,y=g["pos"]

    dirs=[(1,0),(-1,0),(0,1),(0,-1)]
    random.shuffle(dirs)

    best=None
    best_dist=-999 if power_mode else 999

    for dx,dy in dirs:

        nx=x+dx
        ny=y+dy

        if maze[ny][nx]=="1":
            continue

        dist=abs(nx-pacman[0])+abs(ny-pacman[1])

        if power_mode:

            if dist>best_dist:
                best_dist=dist
                best=(nx,ny)

        else:

            if dist<best_dist:
                best_dist=dist
                best=(nx,ny)

    if best:
        g["pos"]=[best[0],best[1]]


def draw_maze():

    for y,row in enumerate(maze):
        for x,col in enumerate(row):

            if col=="1":
                pygame.draw.rect(screen,(0,0,255),(x*CELL,y*CELL,CELL,CELL))

            if (x,y) in pellets:
                pygame.draw.circle(screen,(255,255,255),(x*CELL+CELL//2,y*CELL+CELL//2),4)

            if (x,y) in power_pellets:
                pygame.draw.circle(screen,(255,255,255),(x*CELL+CELL//2,y*CELL+CELL//2),8)


def draw_pacman():

    global mouth_angle,mouth_dir

    center=(pacman[0]*CELL+CELL//2,pacman[1]*CELL+CELL//2)

    mouth_angle+=mouth_dir*4

    if mouth_angle>30 or mouth_angle<5:
        mouth_dir*=-1

    start=math.radians(mouth_angle)
    end=math.radians(360-mouth_angle)

    pygame.draw.circle(screen,(255,255,0),center,12)

    pygame.draw.polygon(screen,(0,0,0),[
        center,
        (center[0]+20*math.cos(start),center[1]+20*math.sin(start)),
        (center[0]+20*math.cos(end),center[1]+20*math.sin(end))
    ])


def draw_ghost(x,y,color):

    px=x*CELL+CELL//2
    py=y*CELL+CELL//2

    pygame.draw.rect(screen,color,(px-12,py-6,24,18))
    pygame.draw.circle(screen,color,(px,py-6),12)

    pygame.draw.circle(screen,(255,255,255),(px-4,py-4),4)
    pygame.draw.circle(screen,(255,255,255),(px+4,py-4),4)

    pygame.draw.circle(screen,(0,0,255),(px-4,py-4),2)
    pygame.draw.circle(screen,(0,0,255),(px+4,py-4),2)


running=True
win=False

while running:

    for event in pygame.event.get():
        if event.type==pygame.QUIT:
            running=False

    move_delay+=1
    ghost_delay+=1

    if move_delay>6:
        move_pacman()
        move_delay=0

    if ghost_delay>18:
        for g in ghosts:
            move_ghost(g)
        ghost_delay=0

    for g in ghosts:

        if pacman==g["pos"]:

            if power_mode:
                g["pos"]=[random.randint(1,18),random.randint(1,9)]
                score+=50
            else:
                running=False

    if not pellets:
        win=True
        running=False

    if power_mode:
        power_timer-=1
        if power_timer<=0:
            power_mode=False

    screen.fill((0,0,0))

    draw_maze()
    draw_pacman()

    for i,g in enumerate(ghosts):

        color=(0,0,255) if power_mode else ghost_colors[i]

        draw_ghost(g["pos"][0],g["pos"][1],color)

    score_text=font.render("Score: "+str(score),True,(255,255,255))
    screen.blit(score_text,(10,10))

    pygame.display.update()

    clock.tick(60)

screen.fill((0,0,0))

if win:
    text=font.render("YOU WIN!",True,(0,255,0))
else:
    text=font.render("GAME OVER",True,(255,0,0))

screen.blit(text,(230,300))
pygame.display.update()

pygame.time.wait(4000)
pygame.quit()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.4)
Hello from the pygame community. https://www.pygame.org/contribute.html
